# Finetune DocNLC tren GPU

Notebook nay tao config YAML rieng cho finetune va chay pipeline co san qua `finetune.py --opt ...`.

Cac file can chuan bi:
- `TRAIN_FILELIST`: moi dong `degraded_image|gt_image`.
- `VAL_FILELIST`: moi dong `degraded_image|gt_image`, co the de rong neu chua validation.
- `PRETRAIN_MODEL_G`: checkpoint `.pth` cua generator da pretrain.
- `OUTPUT_ROOT`: thu muc luu log, model, state va val image.

## 1. Kiem tra moi truong

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import yaml

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "finetune.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "finetune.py").exists():
            PROJECT_ROOT = parent
            break

assert (PROJECT_ROOT / "finetune.py").exists(), "Khong tim thay root repo DocNLC. Hay mo notebook tu thu muc DocNLC."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    for idx in range(torch.cuda.device_count()):
        print(f"GPU {idx}:", torch.cuda.get_device_name(idx))

## 2. Tao filelist neu can

Neu da co filelist 2 cot thi co the bo qua cell nay. Format bat buoc:

```text
/path/to/degraded.png|/path/to/gt.png
```

In [ ]:
IMAGE_EXTS = {".bmp", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def image_map(folder):
    folder = Path(folder).expanduser().resolve()
    assert folder.is_dir(), f"Khong tim thay folder: {folder}"
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    mapping = {}
    for path in files:
        if path.stem in mapping:
            raise ValueError(f"Trung stem {path.stem} trong {folder}")
        mapping[path.stem] = path
    return mapping

def make_pair_filelist(degraded_dir, gt_dir, output_txt, strict=True):
    degraded = image_map(degraded_dir)
    gt = image_map(gt_dir)
    lines = []
    missing = []
    for stem, degraded_path in degraded.items():
        gt_path = gt.get(stem)
        if gt_path is None:
            missing.append(stem)
            continue
        lines.append(f"{degraded_path}|{gt_path}")
    if strict and missing:
        raise RuntimeError(f"Thieu GT cho {len(missing)} anh. Vi du: {missing[:10]}")
    output_txt = Path(output_txt).expanduser().resolve()
    output_txt.parent.mkdir(parents=True, exist_ok=True)
    output_txt.write_text("\n".join(lines) + ("\n" if lines else ""))
    print(f"Da ghi {len(lines)} dong vao {output_txt}")
    if missing:
        print(f"Bo qua {len(missing)} anh bi thieu GT. Vi du:", missing[:10])
    return output_txt

# Vi du su dung, bo comment va sua duong dan neu can:
# TRAIN_FILELIST = make_pair_filelist(
#     degraded_dir=PROJECT_ROOT / "data_finetune" / "train" / "degraded",
#     gt_dir=PROJECT_ROOT / "data_finetune" / "train" / "gt",
#     output_txt=PROJECT_ROOT / "finetune_train.txt",
# )
# VAL_FILELIST = make_pair_filelist(
#     degraded_dir=PROJECT_ROOT / "data_finetune" / "val" / "degraded",
#     gt_dir=PROJECT_ROOT / "data_finetune" / "val" / "gt",
#     output_txt=PROJECT_ROOT / "finetune_val.txt",
# )

## 3. Nhap config finetune

In [ ]:
# Sua cac duong dan nay theo may cua ban.
TRAIN_FILELIST = PROJECT_ROOT / "finetune_train.txt"
VAL_FILELIST = PROJECT_ROOT / "finetune_val.txt"  # de "" neu khong dung validation
PRETRAIN_MODEL_G = PROJECT_ROOT / "pretrain" / "model.pth"

EXPERIMENT_NAME = "DocNLC_finetune_notebook"
OUTPUT_ROOT = PROJECT_ROOT / "output"

GPU_IDS = [0]
BATCH_SIZE = 8
PATCH_SIZE = 256
NITER = 60000
LR_G = 1e-4
LR_STEPS = [30000, 50000]
LR_GAMMA = 0.5
PRINT_FREQ = 40
SAVE_CHECKPOINT_EPOCH = 1
VAL_EPOCH = 1

# Nen de False truoc. Bat True chi khi muon load them teacher net tu cung checkpoint.
DISTILL = False
DISTILL_COFF = 0.2
STRICT_LOAD = False
RESUME_STATE = None

## 4. Validate filelist va checkpoint

In [ ]:
def normalize_optional_path(value):
    if value is None:
        return None
    value = str(value).strip()
    return value or None

def preview_filelist(filelist, expected_cols, name, max_lines=3):
    filelist = Path(filelist).expanduser().resolve()
    assert filelist.exists(), f"Khong tim thay {name}: {filelist}"
    lines = [line.strip() for line in filelist.read_text().splitlines() if line.strip()]
    assert lines, f"{name} dang rong: {filelist}"
    for idx, line in enumerate(lines[:100], 1):
        cols = line.split("|")
        if len(cols) != expected_cols:
            raise ValueError(f"{name} dong {idx} co {len(cols)} cot, can {expected_cols}: {line}")
        missing = [col for col in cols if not Path(col).expanduser().exists()]
        if missing:
            raise FileNotFoundError(f"{name} dong {idx} co path khong ton tai: {missing[0]}")
    print(f"{name}: {filelist} ({len(lines)} dong)")
    for line in lines[:max_lines]:
        print("  ", line)
    return filelist

TRAIN_FILELIST = preview_filelist(TRAIN_FILELIST, 2, "TRAIN_FILELIST")

VAL_FILELIST = normalize_optional_path(VAL_FILELIST)
if VAL_FILELIST:
    VAL_FILELIST = preview_filelist(VAL_FILELIST, 2, "VAL_FILELIST")
else:
    print("VAL_FILELIST rong: notebook se tat validation trong config.")

PRETRAIN_MODEL_G = Path(PRETRAIN_MODEL_G).expanduser().resolve()
assert PRETRAIN_MODEL_G.exists(), f"Khong tim thay PRETRAIN_MODEL_G: {PRETRAIN_MODEL_G}"
print("Checkpoint:", PRETRAIN_MODEL_G)

## 5. Sinh YAML tam cho finetune

In [ ]:
config = {
    "name": EXPERIMENT_NAME,
    "use_tb_logger": True,
    "model": "sr",
    "distortion": "sr",
    "scale": 1,
    "gpu_ids": GPU_IDS,
    "datasets": {
        "train": {
            "name": "UEN",
            "mode": "UEN_train",
            "interval_list": [1],
            "random_reverse": False,
            "border_mode": False,
            "cache_keys": None,
            "filelist": str(TRAIN_FILELIST),
            "use_shuffle": True,
            "n_workers": 0,
            "batch_size": BATCH_SIZE,
            "IN_size": PATCH_SIZE,
            "augment": True,
            "color": "RGB",
        },
    },
    "network_G": {
        "which_model_G": "SID",
        "nf": 16,
        "groups": 8,
    },
    "path": {
        "root": str(OUTPUT_ROOT),
        "results_root": str(OUTPUT_ROOT),
        "pretrain": str(PROJECT_ROOT / "pretrain1"),
        "pretrain_model_G": str(PRETRAIN_MODEL_G),
        "strict_load": STRICT_LOAD,
        "resume_state": RESUME_STATE,
    },
    "train": {
        "lr_G": float(LR_G),
        "lr_scheme": "MultiStepLR",
        "beta1": 0.9,
        "beta2": 0.99,
        "weight_decay_G": 0,
        "niter": int(NITER),
        "ewc": False,
        "distill": bool(DISTILL),
        "ewc_coff": 50.0,
        "distill_coff": float(DISTILL_COFF),
        "fix_some_part": None,
        "warmup_iter": -1,
        "ComputeImportance": False,
        "istraining": True,
        "lr_steps": LR_STEPS,
        "restarts": None,
        "restart_weights": None,
        "clear_state": False,
        "lr_gamma": float(LR_GAMMA),
        "eta_min": 5e-6,
        "pixel_criterion": "l1",
        "pixel_weight": 5000.0,
        "ssim_weight": 1000.0,
        "vgg_weight": 1000.0,
        "val_epoch": int(VAL_EPOCH),
        "manual_seed": 0,
    },
    "logger": {
        "print_freq": int(PRINT_FREQ),
        "save_checkpoint_epoch": int(SAVE_CHECKPOINT_EPOCH),
    },
}

if VAL_FILELIST:
    config["datasets"]["val"] = {
        "name": "UEN",
        "mode": "UEN_test",
        "filelist": str(VAL_FILELIST),
        "batch_size": 1,
        "use_shuffle": False,
        "IN_size": PATCH_SIZE,
    }

generated_dir = PROJECT_ROOT / "options" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = generated_dir / f"{EXPERIMENT_NAME}.yml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

print("Da ghi config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 6. Smoke test config

In [ ]:
from options import options as option
from data import create_dataset

opt = option.parse(str(CONFIG_PATH), is_train=True)
train_set = create_dataset(opt, opt["datasets"]["train"])
sample = train_set[0]
print("Train samples:", len(train_set))
print("Sample keys:", sorted(sample.keys()))
print("LQ shape:", tuple(sample["LQ"].shape), "GT shape:", tuple(sample["GT"].shape))

## 7. Chay finetune

In [ ]:
cmd = [sys.executable, "finetune.py", "--opt", str(CONFIG_PATH), "--launcher", "none"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## 8. Vi tri output

Sau khi chay, ket qua nam trong:

```text
OUTPUT_ROOT/experiments/EXPERIMENT_NAME/
├── models/
├── training_state/
├── val_images/
└── train_EXPERIMENT_NAME.log
```